<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-08/AI_Modul_8.2_Praktikum_Training_Model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 8.2: Praktikum Training Model
**Mata Kuliah:** Kecerdasan Buatan dalam Agrokompleks  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Mengonstruksi arsitektur Jaringan Saraf Tiruan dalam (*Deep Multi-Layer Perceptron*) berbasis `torch.nn.Module`.
2. Merakit siklus pelatihan (*training loop*) kelas industri yang memisahkan fase `model.train()` dan `model.eval()`.
3. Mengintegrasikan teknik pengamanan pemotongan gradien (*Gradient Clipping*) dan penjadwal adaptif `ReduceLROnPlateau`.
4. Mengimplementasikan mekanisme otomatis pencadangan bobot terbaik (*Model Checkpointing*) berbasis kinerja validasi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Konfigurasi seed komputasi untuk reproduktibilitas hasil eksperimen
np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Perangkat Komputasi Aktif: {device}")


## 1. Pembangkitan Data Terpadu & PyTorch DataLoader
Kita melanjutkan dataset agrokompleks terpadu dari Modul 8.7 (600 observasi pohon, 124 fitur masukan, partisi blok spasial bebas kebocoran).


In [ ]:
n_samples = 600
n_spectral = 120
afdeling_ids = np.repeat(np.arange(1, 11), n_samples // 10)

# Fitur Spektral Vis-NIR (120 kanal)
wavelengths = np.linspace(400, 1000, n_spectral)
X_spectral = np.zeros((n_samples, n_spectral))
for i in range(n_samples):
    base_sig = 0.15 + 0.35 * np.exp(-((wavelengths - 550)**2) / (2 * 45**2)) +                       0.50 * (1.0 / (1.0 + np.exp(-(wavelengths - 700) / 20)))
    noise = np.random.normal(0, 0.02, n_spectral)
    X_spectral[i] = base_sig * np.random.uniform(0.85, 1.15) + noise

# Fitur Sensor Tanah & Lingkungan (4 variabel)
pH = np.random.normal(5.2, 0.4, (n_samples, 1))
nitrogen = np.random.normal(120.0, 25.0, (n_samples, 1))
moisture = np.random.uniform(30.0, 70.0, (n_samples, 1))
rainfall = np.random.uniform(100.0, 350.0, (n_samples, 1))
X_sensor = np.hstack([pH, nitrogen, moisture, rainfall])

X_raw = np.hstack([X_spectral, X_sensor])

# Target: Rendemen Minyak CPO (%)
y_raw = 19.5 + 4.5 * X_spectral[:, 65] - 0.015 * np.abs(pH[:, 0] - 5.5)**2 +         0.012 * (nitrogen[:, 0] / 10.0) + np.random.normal(0, 0.35, n_samples)
y_raw = y_raw.reshape(-1, 1)

# Partisi Blok Spasial
train_mask = np.isin(afdeling_ids, [1, 2, 3, 4, 5, 6])
val_mask   = np.isin(afdeling_ids, [7, 8])
test_mask  = np.isin(afdeling_ids, [9, 10])

# Standarisasi Z-Score hanya pada Train
mu_tr = np.mean(X_raw[train_mask], axis=0, keepdims=True)
std_tr = np.std(X_raw[train_mask], axis=0, keepdims=True) + 1e-8

X_train_norm = (X_raw[train_mask] - mu_tr) / std_tr
X_val_norm   = (X_raw[val_mask] - mu_tr) / std_tr
X_test_norm  = (X_raw[test_mask] - mu_tr) / std_tr

y_train = y_raw[train_mask]
y_val   = y_raw[val_mask]
y_test  = y_raw[test_mask]

# Kelas Dataset
class AgroDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X = torch.tensor(X_data, dtype=torch.float32)
        self.y = torch.tensor(y_data, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(AgroDataset(X_train_norm, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(AgroDataset(X_val_norm, y_val), batch_size=32, shuffle=False)
test_loader  = DataLoader(AgroDataset(X_test_norm, y_test), batch_size=32, shuffle=False)

print(f"Data latih siap: {len(train_loader.dataset)} sampel.")
print(f"Data validasi  : {len(val_loader.dataset)} sampel.")
print(f"Data uji       : {len(test_loader.dataset)} sampel.")


## 2. Definisi Arsitektur Jaringan Saraf Tiruan (Deep MLP)
Kita mengonstruksi jaringan saraf 3-lapisan [124, 64, 32, 1] dengan Batch Normalization dan Dropout terintegrasi.


In [ ]:
class OilPalmYieldMLP(nn.Module):
    def __init__(self, in_features=124, hidden1=64, hidden2=32, out_features=1):
        super(OilPalmYieldMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(p=0.15),
            
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(p=0.10),
            
            nn.Linear(hidden2, out_features)
        )
        
    def forward(self, x):
        return self.net(x)

# Instansiasi model
model = OilPalmYieldMLP().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameter Bebas: {total_params:,}")


## 3. Implementasi Kelas Orkestrator Pelatihan (Training Pipeline)
Kelas `DeepLearningTrainer` mengelola seluruh siklus pelatihan:
- Pemisahan fase `model.train()` dan `model.eval()`.
- Pengamanan gradien melalui `torch.nn.utils.clip_grad_norm_`.
- Penjadwalan laju pembelajaran adaptif menggunakan `ReduceLROnPlateau`.
- Pencadangan otomatis bobot terbaik (*Best Checkpointing*).


In [ ]:
class DeepLearningTrainer:
    def __init__(self, model, train_loader, val_loader, lr=0.003, weight_decay=1e-4, max_norm=1.0):
        self.device = device
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.max_norm = max_norm
        
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='min', factor=0.5, patience=4)
        
        self.history = {'train_loss': [], 'val_loss': [], 'lr': []}
        self.best_val_loss = float('inf')
        self.checkpoint_path = 'best_oilpalm_model.pt'

    def train_epoch(self):
        self.model.train()
        running_loss = 0.0
        for batch_x, batch_y in self.train_loader:
            batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
            
            self.optimizer.zero_grad()
            preds = self.model(batch_x)
            loss = self.criterion(preds, batch_y)
            loss.backward()
            
            # Pemotongan Gradien Global
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.max_norm)
            self.optimizer.step()
            
            running_loss += loss.item() * batch_x.size(0)
        return running_loss / len(self.train_loader.dataset)

    def validate_epoch(self):
        self.model.eval()
        running_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in self.val_loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                preds = self.model(batch_x)
                loss = self.criterion(preds, batch_y)
                running_loss += loss.item() * batch_x.size(0)
        return running_loss / len(self.val_loader.dataset)

    def fit(self, num_epochs=60):
        print("=================================================================")
        print(f"      MEMULAI PELATIHAN TRAINING PIPELINE (TOTAL {num_epochs} EPOCH)     ")
        print("=================================================================")
        for epoch in range(1, num_epochs + 1):
            tr_loss = self.train_epoch()
            va_loss = self.validate_epoch()
            
            self.scheduler.step(va_loss)
            curr_lr = self.optimizer.param_groups[0]['lr']
            
            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(va_loss)
            self.history['lr'].append(curr_lr)
            
            if va_loss < self.best_val_loss:
                self.best_val_loss = va_loss
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_loss': self.best_val_loss
                }, self.checkpoint_path)
                status_save = "--> [MODEL CHECKPOINT TERSIMPAN]"
            else:
                status_save = ""
                
            if epoch % 10 == 0 or epoch == 1 or status_save:
                print(f"Epoch [{epoch:02d}/{num_epochs:02d}] | Train MSE: {tr_loss:.4f} | Val MSE: {va_loss:.4f} | LR: {curr_lr:.6f} {status_save}")
                
        print("=================================================================")
        print(f"Pelatihan Selesai! Model terbaik tersimpan dengan Val MSE: {self.best_val_loss:.4f}")
        return self.history


## 4. Eksekusi Pelatihan Model
Kita menjalankan proses pelatihan selama 50 epoch dan memantau peluruhan laju belajar serta pencadangan checkpoint.


In [ ]:
trainer = DeepLearningTrainer(model, train_loader, val_loader, lr=0.005, max_norm=1.0)
history = trainer.fit(num_epochs=50)


## 5. Visualisasi Kurva Konvergensi dan Dinamika Learning Rate
Kita menampilkan grafik kurva belajar (Train vs Val Loss) serta riwayat adaptasi laju pembelajaran.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Kurva Evaluasi Loss
epochs_range = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs_range, history['train_loss'], 'b-', lw=1.8, label='Train MSE Loss')
ax1.plot(epochs_range, history['val_loss'], 'r--', lw=1.8, label='Val MSE Loss')
ax1.set_title('Dinamika Konvergensi Pelatihan Model Deep MLP', fontsize=11, fontweight='bold')
ax1.set_xlabel('Epoch Pelatihan')
ax1.set_ylabel('Mean Squared Error (MSE)')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# Plot 2: Riwayat Learning Rate
ax2.plot(epochs_range, history['lr'], color='darkorange', lw=2, marker='o', markersize=4)
ax2.set_title('Penurunan Dinamis Laju Belajar (ReduceLROnPlateau)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Epoch Pelatihan')
ax2.set_ylabel('Learning Rate (Skala Log)')
ax2.set_yscale('log')
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## 6. Pemulihan Bobot Terbaik (*Checkpoint Restoration*) & Evaluasi Data Uji
Kita memulihkan bobot terbaik yang tersimpan pada `best_oilpalm_model.pt` dan menguji reliabilitas generalisasinya pada himpunan data uji independen.


In [ ]:
# Muat bobot checkpoint terbaik
checkpoint = torch.load('best_oilpalm_model.pt')
best_model = OilPalmYieldMLP().to(device)
best_model.load_state_dict(checkpoint['model_state_dict'])
best_model.eval()

print(f"Checkpoint terbaik berhasil dipulihkan dari Epoch ke-{checkpoint['epoch']}")
print(f"Validation MSE saat checkpoint: {checkpoint['val_loss']:.4f}")

# Evaluasi pada data uji (Test Set)
test_loss = 0.0
all_preds = []
all_trues = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        preds = best_model(batch_x)
        loss = nn.MSELoss()(preds, batch_y)
        test_loss += loss.item() * batch_x.size(0)
        
        all_preds.extend(preds.cpu().numpy().flatten())
        all_trues.extend(batch_y.cpu().numpy().flatten())

avg_test_loss = test_loss / len(test_loader.dataset)
r2_score = 1.0 - (np.sum((np.array(all_trues) - np.array(all_preds))**2) / 
                  np.sum((np.array(all_trues) - np.mean(all_trues))**2))

print("=================================================================")
print("             HASIL EVALUASI MODEL TERBAIK PADA DATA UJI          ")
print("=================================================================")
print(f"Mean Squared Error (MSE) Data Uji : {avg_test_loss:.4f}")
print(f"Koefisien Determinasi (R2) Uji    : {r2_score:.4f}")
print("=================================================================")
print("[OK] Alur Pelatihan (Training Pipeline) berhasil beroperasi secara tangguh!")
